# Valiron — End-to-End Demo
# Tests all modules: evaluate, subgroups,
# calibration, ACP, monitoring, report
# pip installs automatically — just Run All
# github.com/abhaysachan007/valiron

In [1]:
# Cell 1: Install valiron
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'valiron', '--quiet'], capture_output=True)
import valiron
print(f'\u2705 valiron installed — version: {valiron.__version__}')

✅ valiron installed — version: 0.1.0


In [2]:
# Cell 2: Synthetic clinical dataset (balanced 50/50)
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
N = 300
glucose = rng.normal(120, 30, N).clip(70, 200)
bmi     = rng.normal(30, 6, N).clip(18, 50)
age     = rng.integers(25, 80, N).astype(float)
insulin = rng.normal(80, 40, N).clip(10, 300)
bp      = rng.normal(75, 12, N).clip(50, 120)
# Balanced via median split — exactly 50/50, strong signal, specificity passes 0.60
score = 0.04*glucose + 0.08*bmi + 0.01*age - 0.002*insulin + 0.01*bp
y     = (score > np.median(score)).astype(int)
X = np.column_stack([glucose, bmi, age, insulin, bp])
feature_names = ['glucose', 'bmi', 'age', 'insulin', 'blood_pressure']
age_bins     = pd.cut(age, bins=[0,35,55,100], labels=['18-35','36-55','55+'])
gender       = rng.choice(['M','F'], N)
site         = rng.choice(['Delhi','Mumbai','Chennai'], N)
sensitive_df = pd.DataFrame({'age_group': age_bins, 'gender': gender, 'site': site})
X_train, X_test, y_train, y_test, _, sens_test = train_test_split(
    X, y, sensitive_df, test_size=0.33, random_state=42
)
sens_test = sens_test.reset_index(drop=True)
print(f'\u2705 Dataset — {N} samples | train: {len(X_train)} | test: {len(X_test)} | prevalence: {y.mean():.1%}')

✅ Dataset — 300 samples | train: 201 | test: 99 | prevalence: 50.0%


In [3]:
# Cell 3: Train model
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(f'\u2705 Model trained — train acc: {model.score(X_train, y_train):.3f} | test acc: {model.score(X_test, y_test):.3f}')

✅ Model trained — train acc: 1.000 | test acc: 0.919


In [4]:
# Cell 4: Metrics with bootstrap CIs
from valiron.evaluate.metrics import compute_metrics
metrics = compute_metrics(y_test, y_pred, y_prob, bootstrap_n=500, seed=42)
def _ci(t): return f'[{t[0]:.3f}-{t[1]:.3f}]' if t else 'n/a'
print('\u2705 Metrics (95% bootstrap CI):')
print(f'   Accuracy:    {metrics.accuracy:.3f} {_ci(metrics.accuracy_ci)}')
print(f'   Sensitivity: {metrics.sensitivity:.3f} {_ci(metrics.sensitivity_ci)}')
print(f'   Specificity: {metrics.specificity:.3f} {_ci(metrics.specificity_ci)}')
print(f'   F1:          {metrics.f1:.3f} {_ci(metrics.f1_ci)}')
print(f'   AUC-ROC:     {metrics.auc:.3f} {_ci(metrics.auc_ci)}')

✅ Metrics (95% bootstrap CI):
   Accuracy:    0.919 [0.859-0.970]
   Sensitivity: 0.959 [0.895-1.000]
   Specificity: 0.880 [0.770-0.971]
   F1:          0.922 [0.864-0.970]
   AUC-ROC:     0.980 [0.954-0.997]


In [5]:
# Cell 5: CDSCO compliance
eval_result = valiron.evaluate(
    model=model, X_test=X_test, y_test=y_test,
    regulation='cdsco_mdsw', use_case='diagnostic_imaging_aid',
    sensitive_features=['gender', 'age_group'], feature_names=feature_names,
)
status = '\u2705 COMPLIANT' if eval_result.compliant else '\u274c NOT COMPLIANT'
print(f'{status} — CDSCO MDSW  |  score={eval_result.score:.3f}')
print(f'   Passing: {len(eval_result.passing_checks)}  Failing: {len(eval_result.failing_checks)}')
for c in eval_result.failing_checks: print(f'   \u26a0 {c}')

✅ COMPLIANT — CDSCO MDSW  |  score=1.000
   Passing: 4  Failing: 0


In [6]:
# Cell 6: Subgroup fairness
from valiron.subgroups.analyzer import analyze_subgroups, disparity_report
subgroups = analyze_subgroups(y_test, y_pred, sens_test)
disparity = disparity_report(y_test, y_pred, sens_test, metric='accuracy')
n_groups = sum(len(v) for v in subgroups.values())
max_disp = max(v['accuracy_disparity'] for v in disparity.values())
print(f'\u2705 Subgroups: {n_groups} groups across {len(subgroups)} features | max disparity: {max_disp:.3f}')
for feat, rep in disparity.items():
    print(f"   {feat}: disp={rep['accuracy_disparity']:.3f} worst={rep['worst_group']} best={rep['best_group']}")

✅ Subgroups: 8 groups across 3 features | max disparity: 0.085
   age_group: disp=0.010 worst=18-35 best=55+
   gender: disp=0.035 worst=F best=M
   site: disp=0.085 worst=Delhi best=Chennai


In [7]:
# Cell 7: Calibration
from valiron.calibration.checker import check_calibration
cal  = check_calibration(y_test, y_prob, n_bins=10)
icon = '\u2705' if cal.well_calibrated else '\u26a0'
print(f'{icon} Calibration: ECE={cal.ece:.4f} (threshold<0.10) | MCE={cal.mce:.4f} | well_calibrated={cal.well_calibrated}')

✅ Calibration: ECE=0.0591 (threshold<0.10) | MCE=0.5408 | well_calibrated=True


In [8]:
# Cell 8: ACP — save two versions, diff them
from valiron.acp import save_version, diff_versions, generate_acp_document
def _sg_flat(sg):
    result = {}
    for feat, groups in sg.items():
        result[feat] = {}
        for grp, r in groups.items():
            if hasattr(r, 'accuracy'):
                result[feat][str(grp)] = {'accuracy': r.accuracy, 'f1': r.f1,
                                           'sensitivity': r.sensitivity, 'specificity': r.specificity}
            else:
                result[feat][str(grp)] = dict(r)
    return result
v1 = save_version(model, {
    'id': 'v1-baseline', 'name': 'DiabetesModel', 'version_string': '1.0.0',
    'metrics': metrics, 'subgroups': _sg_flat(subgroups),
    'regulation': 'cdsco_mdsw', 'notes': 'Baseline GBC n_estimators=100',
})
model2     = GradientBoostingClassifier(n_estimators=120, max_depth=4, random_state=7)
model2.fit(X_train, y_train)
y_pred2    = model2.predict(X_test)
y_prob2    = model2.predict_proba(X_test)[:, 1]
metrics2   = compute_metrics(y_test, y_pred2, y_prob2, bootstrap_n=0, seed=0)
subgroups2 = analyze_subgroups(y_test, y_pred2, sens_test)
v2 = save_version(model2, {
    'id': 'v2-candidate', 'name': 'DiabetesModel', 'version_string': '1.1.0',
    'metrics': metrics2, 'subgroups': _sg_flat(subgroups2),
    'regulation': 'cdsco_mdsw', 'notes': 'Candidate: n_estimators=120 max_depth=4',
})
diff    = diff_versions(v1, v2)
acp_doc = generate_acp_document(diff)
rec     = next((l for l in acp_doc.split('\n') if any(x in l for x in ['**APPROVE**','**REVIEW**','**REJECT**'])), 'unknown')
print(f'\u2705 ACP diff v1->v2 | regression={diff.regression_detected}')
for m, d in diff.metric_changes.items():
    print(f'   {m}: {d:+.4f}')
print(f'   {rec}')

✅ ACP diff v1->v2 | regression=False
   accuracy: +0.0303
   f1: +0.0289
   sensitivity: +0.0204
   specificity: +0.0400
   **APPROVE** — no regressions detected, candidate meets performance baseline.


In [9]:
# Cell 9: Drift monitoring
from valiron.monitoring import monitor
baseline_df   = pd.DataFrame(X_train, columns=feature_names)
current_df    = pd.DataFrame(X_test,  columns=feature_names)
train_metrics = compute_metrics(y_train, model.predict(X_train),
                                model.predict_proba(X_train)[:,1], bootstrap_n=0, seed=0)
drift = monitor(baseline_df, current_df, train_metrics, metrics)
icon = '\u26a0' if drift.drift_detected else '\u2705'
print(f'{icon} Drift: severity={drift.severity} | detected={drift.drift_detected}')
for feat, psi in sorted(drift.psi_scores.items(), key=lambda x: -x[1])[:3]:
    level = 'SEVERE' if psi>=0.2 else ('MILD' if psi>=0.1 else 'OK')
    print(f'   {feat}: PSI={psi:.4f} [{level}]')
for a in drift.alerts: print(f'   \u26a0 {a}')

⚠ Drift: severity=severe | detected=True
   blood_pressure: PSI=0.3661 [SEVERE]
   insulin: PSI=0.2673 [SEVERE]
   bmi: PSI=0.1670 [MILD]
   ⚠ PERFORMANCE: accuracy dropped 8.1% (baseline=1.0000, current=0.9192)
   ⚠ PERFORMANCE: f1 dropped 7.8% (baseline=1.0000, current=0.9216)
   ⚠ PERFORMANCE: specificity dropped 12.0% (baseline=1.0000, current=0.8800)


In [10]:
# Cell 10: CDSCO HTML report
from valiron.report.builder import ReportInput, report
from pathlib import Path
ri       = ReportInput(eval_result=eval_result, metrics=metrics, subgroups=subgroups, calibration=cal)
out_path = Path('valiron_cdsco_report.html')
report(ri, format='html', output=str(out_path))
print(f'\u2705 CDSCO HTML report: {out_path} ({out_path.stat().st_size/1024:.1f} KB)')

✅ CDSCO HTML report: valiron_cdsco_report.html (11.7 KB)


In [11]:
# Cell 11: Summary
from pathlib import Path
report_ok = Path('valiron_cdsco_report.html').exists()
acp_v1_ok = Path('.valiron/v1-baseline.json').exists()
acp_v2_ok = Path('.valiron/v2-candidate.json').exists()
print('=' * 52)
print('  VALIRON END-TO-END VERIFICATION')
print('=' * 52)
print(f'  \u2705 valiron installed         v{valiron.__version__}')
print(f'  \u2705 dataset (N={N})           synthetic, 5 features')
print(f'  \u2705 model trained             GradientBoostingClassifier')
print(f'  \u2705 metrics + CIs             accuracy={metrics.accuracy:.3f}')
print(f'  \u2705 CDSCO compliance          compliant={eval_result.compliant}')
print(f'  \u2705 subgroup analysis         {n_groups} groups, max_disp={max_disp:.3f}')
print(f'  \u2705 calibration               ECE={cal.ece:.4f}')
print(f'  \u2705 ACP diff v1->v2           regression={diff.regression_detected}')
print(f'  \u2705 drift monitor             severity={drift.severity}')
print(f'  \u2705 HTML report               exists={report_ok}')
print('=' * 52)
print()
print('\U0001f389 Valiron working end-to-end')

  VALIRON END-TO-END VERIFICATION
  ✅ valiron installed         v0.1.0
  ✅ dataset (N=300)           synthetic, 5 features
  ✅ model trained             GradientBoostingClassifier
  ✅ metrics + CIs             accuracy=0.919
  ✅ CDSCO compliance          compliant=True
  ✅ subgroup analysis         8 groups, max_disp=0.085
  ✅ calibration               ECE=0.0591
  ✅ ACP diff v1->v2           regression=False
  ✅ drift monitor             severity=severe
  ✅ HTML report               exists=True

🎉 Valiron working end-to-end


## Results
- All 6 modules working
- CDSCO compliant report generated
- `pip install valiron` to use in your project
- Star us: github.com/abhaysachan007/valiron